# Tokenizer: one artifact, declared and then bound

`tokenizers.bpe` is one tokenizer family, whole: the artifact, the jobs that
build it, and the BPE internals both share. `Tokenizer` is a dag Artifact and
the tokenizer itself, in one object:

- as an **artifact**, it's the parameters of a training run (vocab_size,
  special_tokens, which sources) plus the folder it owns. You can write one
  down, hash it, declare it, and put it in a manifest before anything has
  been trained.
- as a **tokenizer**, it encodes and decodes text -- but only once it holds
  the vocab and merges a finished run produced. Those aren't parameters (they
  are what the job *wrote*), so they arrive through `bind`:
  `tokenizer.bind(root)` reads them back out of the folder, and
  `tokenizer.bind(vocab=..., merges=...)` takes them straight from the job
  that just computed them.

This notebook walks the whole path -- declare, build, bind, encode --
against a local folder, with no Modal involved.

In [1]:
import logging
from pathlib import Path
from types import SimpleNamespace

from dag import resolve as dag_resolve
from sources.artifact import Source
from tokenizers.bpe import TokenizedSource, Tokenizer

# A throwaway stand-in for the Modal volume. On Modal, root is Path(STORAGE)
# and jobs are driven by main.py's run_job; nothing in the artifact/job layer
# knows about either -- a job takes a root and a worker, and that's all.
ROOT = Path(".scratch/demo-volume").resolve()
ROOT.mkdir(parents=True, exist_ok=True)

# A job only ever reaches for worker.log, so a namespace with a logger stands
# in for the real runtime.Worker (which carries a lease and a call id it has
# nothing to lease against here).
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
worker = SimpleNamespace(log=logging.getLogger("demo"))

ROOT

PosixPath('/Users/oguz/Projects/launchpad/.scratch/demo-volume')

## 1. Declaring one

Written by hand, same as in [declare.ipynb](declare.ipynb). It answers
*which* tokenizer this is and *where* it lives -- and nothing about bytes,
because nothing has been trained yet.

In [2]:
romeojuliet = Source(
    name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt"
)

tokenizer = Tokenizer(
    vocab_size=1000,
    special_tokens=("<|endoftext|>",),
    sources=(romeojuliet,),
)

print(tokenizer.uid)  # readable id, derived from the parameters
print(tokenizer.artifact_path)  # the folder it owns, relative to root
print(tokenizer.paths(ROOT)["tokenizer"])  # where its one file will land
print(tokenizer.exists(ROOT))  # nothing built yet

bpe-1000-feeeeefa90
tokenizers/bpe-1000-feeeeefa90
/Users/oguz/Projects/launchpad/.scratch/demo-volume/tokenizers/bpe-1000-feeeeefa90/tokenizer.json
False


In [3]:
# So it can't tokenize anything yet: there is no trained state to bind, and
# nothing to read it from. Both ways of asking say so rather than handing back
# something half-formed.
try:
    tokenizer.encode("But soft")
except RuntimeError as err:
    print("not bound ->", err)

try:
    tokenizer.bind(ROOT)
except FileNotFoundError as err:
    print("not built ->", err)

not bound -> bpe-1000-feeeeefa90 has no vocab or merges yet -- bind(root) to read the tokenizer.json its job wrote, or bind(vocab=..., merges=...), before encoding
not built -> [Errno 2] No such file or directory: '/Users/oguz/Projects/launchpad/.scratch/demo-volume/tokenizers/bpe-1000-feeeeefa90/tokenizer.json'


## 2. Declare, then build

`TokenizedSource` is asked for rather than the tokenizer itself, to show the
tokenizer being used as a dependency: it's romeojuliet encoded *by this
tokenizer*, so resolving it pulls in the source and the tokenizer beneath it.

In [4]:
tokenized = TokenizedSource(tokenizer=tokenizer, source=romeojuliet)

declaration = dag_resolve.Declaration(tokenized, ROOT).check()
declaration  # reads the disk, writes nothing

run - under /Users/oguz/Projects/launchpad/.scratch/demo-volume
  new        sources/romeojuliet
  new        tokenizers/bpe-1000-feeeeefa90
  new        tokenizers/bpe-1000-feeeeefa90/bin/romeojuliet

3 new
ok -- 3 to declare

In [5]:
declaration.write()  # one manifest.json per artifact, dependencies first

[PosixPath('/Users/oguz/Projects/launchpad/.scratch/demo-volume/sources/romeojuliet/manifest.json'),
 PosixPath('/Users/oguz/Projects/launchpad/.scratch/demo-volume/tokenizers/bpe-1000-feeeeefa90/manifest.json'),
 PosixPath('/Users/oguz/Projects/launchpad/.scratch/demo-volume/tokenizers/bpe-1000-feeeeefa90/bin/romeojuliet/manifest.json')]

In [6]:
# Run the plan in dependency order: download the source, train the tokenizer,
# then encode the source with it. Re-running the cell skips what's done, so
# only the missing pieces are rebuilt.
for job in dag_resolve.job_list(dag_resolve.resolve(tokenized)):
    if dag_resolve.status(job.artifact, ROOT) == "done":
        print(f"already done: {job.artifact.artifact_path}")
        continue
    job.run(ROOT, worker)

downloading romeojuliet from https://www.gutenberg.org/cache/epub/1513/pg1513.txt


wrote 167469 chars for romeojuliet


training BPE tokenizer (vocab_size=1000) on 1 source(s)


pretokenizing and building frequency map:   0%|          | 0.00/170k [00:00<?, ?B/s]

pretokenizing and building frequency map: 100%|██████████| 170k/170k [00:00<00:00, 12.0MB/s]

...

merging pairs:  96%|█████████▌| 714/743 [00:00<00:00, 939.99it/s]

merging pairs: 100%|██████████| 743/743 [00:00<00:00, 890.42it/s]


trained, vocab has 1000 entries


tokenizing romeojuliet


wrote 65754 tokens for romeojuliet


## 3. Binding the trained state

The folder is filled in now, so the same object can be handed its vocab and
merges and start tokenizing. `bind` returns the artifact itself -- it isn't a
different object, and binding changes nothing about which artifact it is (the
state is not a parameter, so it stays out of `==`, `hash`, and the manifest).

In [7]:
tokenizer.bind(ROOT)

print(f"{len(tokenizer.vocab)} vocab entries, {len(tokenizer.merges)} merges")
print("first merges:", [a + b for a, b in tokenizer.merges[:8]])
print("specials:", tokenizer.special_tokens)  # a parameter, not something loaded

# the same tokenizer, spelled out again: still equal, still the same folder
print(Tokenizer(vocab_size=1000, special_tokens=("<|endoftext|>",), sources=(romeojuliet,)) == tokenizer)

1000 vocab entries, 743 merges
first merges: [b' t', b'he', b' a', b' s', b'ou', b'in', b' w', b' m']
specials: ('<|endoftext|>',)
True


In [8]:
line = "But soft, what light through yonder window breaks?<|endoftext|>"

ids = tokenizer.encode(line)
print(ids)
print([tokenizer.vocab[i] for i in ids])  # what each id stands for
print(repr(tokenizer.decode(ids)))  # round-trips, special token included

[487, 380, 102, 116, 44, 542, 711, 285, 114, 854, 296, 111, 814, 262, 520, 310, 756, 97, 485, 63, 999]
[b'But', b' so', b'f', b't', b',', b' what', b' light', b' th', b'r', b'ough', b' y', b'o', b'nder', b' w', b'ind', b'ow', b' bre', b'a', b'ks', b'?', b'<|endoftext|>']
'But soft, what light through yonder window breaks?<|endoftext|>'


The same object read the other way round: `TokenizedSource` holds this
source already encoded, and the tokenizer that encoded it is what turns those
ids back into text.

In [9]:
token_ids = [int(t) for t in tokenized.paths(ROOT)["tokens"].read_text().split()]

print(f"{len(token_ids)} tokens on disk at {tokenized.artifact_path}")
print(repr(tokenizer.decode(token_ids[:80])))

65754 tokens on disk at tokenizers/bpe-1000-feeeeefa90/bin/romeojuliet
'The Project Gutenberg eBook of Romeo and Juliet\n    \nThis eBook is for the use of anyone anywhere in the United States and\nmost other parts of the world at no cost and with almost no restrictions\nwhatsoever. You may copy it, give it away or re-'


## The other direction

`bind` has a second form and an inverse. The job that trains a tokenizer
doesn't build some other object and hand it over -- it binds the artifact it
was given to the vocab and merges it just computed, and tells it to write
itself out. That's the last line of `TokenizerJob.run`, a few classes down
the same module ([tokenizers/bpe.py](tokenizers/bpe.py)):

```python
self.artifact.bind(vocab=vocab, merges=merges).save(root)
```

So the whole life of a tokenizer is three calls on one object:

| | call | who does it |
| --- | --- | --- |
| trained state -> its folder | `tokenizer.bind(vocab=..., merges=...).save(root)` | the training job, once |
| its folder -> trained state | `tokenizer.bind(root)` | everyone downstream |
| use it | `tokenizer.encode(...)` / `tokenizer.decode(...)` | anyone, once bound |

`save` writes into the folder the artifact owns, and `bind(root)` reads back
from it and checks the file agrees with what this artifact declares -- a
tokenizer.json whose special tokens aren't these raises instead of quietly
encoding with someone else's vocab.

Nothing above changes on Modal: `ROOT` becomes `Path(STORAGE)`, `worker`
becomes the real `runtime.Worker`, and the calls read the same.

In [10]:
# Cleanup, if you want the demo volume gone:
# import shutil; shutil.rmtree(ROOT)